# Train the satellite nowcasting U-Net on a Colab GPU

Uses the daily zips produced by `colab_himawari.ipynb` (in `MyDrive/stormcast/himawari_days/`):
builds the Himawari-only gold Zarr, trains the U-Net and compares it with persistence and optical flow.

**Runtime → Change runtime type → T4 GPU** before running. Results (metrics.json, skill plot, weights)
are saved to `MyDrive/stormcast/runs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/stormcast'

In [ ]:
%%bash -s "$DRIVE_DIR"
set -e
rm -rf /content/stormcast && mkdir -p /content/stormcast /content/data
unzip -q "$1/stormcast-code.zip" -d /content/stormcast
for z in "$1"/himawari_days/himawari_*.zip; do unzip -q -o "$z" -d /content/data; done
cd /content/stormcast && pip install -q -e ".[ml]"
ls /content/data/silver/himawari | wc -l

In [ ]:
import os, yaml, subprocess
os.chdir('/content/stormcast')
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['paths']['data_root'] = '/content/data'
cfg['ml']['sat_zarr_path'] = '/content/data/gold/himawari_aug2025.zarr'
cfg['ml']['output_dir'] = f'{DRIVE_DIR}/runs'
cfg['ml']['num_workers'] = 2
yaml.safe_dump(cfg, open('configs/colab.yaml', 'w'))
!stormcast --config configs/colab.yaml build-sat
!stormcast --config configs/colab.yaml report && cat reports/data_quality.md

In [ ]:
# quick check (1 epoch, few batches) before the real run
!python -m stormcast.ml.train_nowcast --config configs/colab.yaml --epochs 1 --max-train-batches 10

In [ ]:
!python -m stormcast.ml.train_nowcast --config configs/colab.yaml

In [ ]:
import glob, json
from IPython.display import Image, display
run = sorted(glob.glob(f'{DRIVE_DIR}/runs/nowcast-*'))[-1]
display(Image(f'{run}/skill_vs_lead.png'))
test = json.load(open(f'{run}/metrics.json'))['test']
for method, per_lead in test.items():
    print(f'{method:13s}', '  '.join(f"{k}: {v['csi@235K']:.3f}" for k, v in per_lead.items()))